In [5]:
import unittest
import pandas as pd
import sqlalchemy
from sqlalchemy import create_engine, Column, Integer, String, Float, ForeignKey, text, insert
from pandas.testing import assert_frame_equal

# Putanja do predprocesirane CSV datoteke
CSV_FILE_PATH = "Support_tickets_PROCESSED.csv"

In [8]:
# 1. Postavke povezivanja
USER = 'root'
PASSWORD = '3k0p13!4'
HOST = 'localhost'
DB_NAME = 'fipu_srp_projekt'

# Kreiranje engine-a
engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}/{DB_NAME}")

# 2. Provjera broja redaka
# Broj redaka u CSV datoteci
df_processed = pd.read_csv(CSV_FILE_PATH)
csv_count = len(df_processed)
  
# Broj redaka u SQL tablici
with engine.connect() as connection:
    result = connection.execute(text("SELECT COUNT(*) FROM support_tickets"))
    db_count = result.scalar()
        
print(f"--- PROVJERA KOLIČINE PODATAKA ---")
print(f"Broj redaka u CSV datoteci: {csv_count}")
print(f"Broj redaka u SQL tablici: {db_count}")
   
if csv_count == db_count:
    print("Broj redaka se podudara.")
else:
    print("Broj redaka nije identičan!")


--- PROVJERA KOLIČINE PODATAKA ---
Broj redaka u CSV datoteci: 53353
Broj redaka u SQL tablici: 53353
Broj redaka se podudara.


In [9]:
# 3. Provjera uzorka podataka i deanonimiziranih stupaca
query = "SELECT * FROM support_tickets LIMIT 5"
df_sample = pd.read_sql(query, engine)
    
print("\n--- UZORAK PODATAKA IZ BAZE ---")
# Provjeravamo jesu li naši novi stupci (projekt i tehničari) prisutni
cols_to_show = ['ticket_id', 'issue_proj_name', 'issue_reporter_name', 'issue_assignee_name']
   
# Prikazujemo samo bitne stupce ako postoje, inače cijeli sample
available_cols = [c for c in cols_to_show if c in df_sample.columns]
print(df_sample[available_cols if available_cols else df_sample.columns])


--- UZORAK PODATAKA IZ BAZE ---
        id                    started                      ended  issue_num  \
0  11887.0  2016-01-06 08:23:43+00:00  2016-01-06 08:56:55+00:00      186.0   
1  11890.0  2016-01-11 10:06:19+00:00  2016-01-12 12:30:23+00:00      190.0   
2  11904.0  2016-01-21 07:28:20+00:00  2016-01-26 08:21:47+00:00      198.0   
3  11907.0  2016-01-26 07:44:54+00:00  2016-01-26 07:45:48+00:00      209.0   
4  11912.0  2016-02-01 13:45:47+00:00  2016-02-07 06:21:42+00:00      217.0   

      issue_proj  issue_reporter issue_assignee  issue_contr_count issue_type  \
0  Project Atlas  Petra Pavlović           None                1.0     Ticket   
1  Project Atlas  Petra Pavlović           None                1.0     Ticket   
2  Project Atlas      Luka Božić      Ana Božić                1.0     Ticket   
3  Project Atlas  Petra Pavlović           None                1.0   Vacation   
4  Project Atlas      Luka Božić      Ana Božić                1.0      Story   

  iss